# Pass prediction with propygator

This notebook walks through **Feature 1.5**: given a satellite and a ground station, find
the **visible passes** — when the satellite is above the horizon, sunlit, and the observer
is in darkness — with an estimated **visual magnitude**, then turn them into a table, a
calendar file, and two plots.

Feature 1.5 is composition over shipped primitives — `propagate_tle` and the
`look_angles_track` topocentric kernel — plus two pieces of new physics in
`tracking/visibility.py`: a conical-umbra shadow test and a phase-law magnitude. As
everywhere in propygator, building the `TLE` and the value types is pure-Python; the JVM
starts lazily at the first propagation (`docs/architecture.md` §10).

We use a **pinned ISS element set and a fixed search window** so the notebook runs offline
and reproducibly. The commented lines further down show the "tonight, live" variant.

In [ ]:
import propygator as pgr

pgr.__version__

## 1. A satellite, a station, and a window

`find_passes` needs a `TLE`, a `GroundStation`, and a search window. We pin a historical
ISS element set and a fixed `start` so the results below are deterministic. For real
"tonight" predictions, fetch the current element set and drop the explicit `start` (it
defaults to `Epoch.now()`):

```python
tle = pgr.fetch_tle("ISS")                     # current element set from CelesTrak (cached)
passes = pgr.find_passes(tle, durham, 86400)   # next 24 h, from now
```

In [ ]:
# A pinned ISS (ZARYA) element set (epoch 2026-06-20) + the README's Durham, NC station.
ISS_LINE1 = "1 25544U 98067A   26171.41461525  .00008813  00000+0  16600-3 0  9990"
ISS_LINE2 = "2 25544  51.6327 284.1189 0004557 208.5194 151.5545 15.49333088572250"

tle = pgr.TLE.from_strings(ISS_LINE1, ISS_LINE2, name="ISS (ZARYA)")
durham = pgr.GroundStation("Durham", 35.99, -78.90, altitude_m=130)
start = pgr.Epoch.from_iso("2026-06-23T00:00:00", scale=pgr.TimeScale.UTC)

print(tle.name, "| NORAD", tle.norad_id)

## 2. Find the visible passes

`find_passes(tle, station, duration, *, start=None, min_elevation_deg=10.0,
visible_only=True, standard_magnitude=None, progress=True)` scans the window, refines each
pass's rise / culmination / set to sub-second precision, and annotates it.

- A **pass** is a maximal interval with elevation ≥ `min_elevation_deg` (default 10°, the
  usual visual-observing threshold), so `rise` / `set` are crossings of *that gate*, not
  the 0° horizon.
- A pass is **visible** if at some point the satellite is sunlit *and* the station is in
  darkness (Sun ≤ −6°). `visible_only=True` (the default) keeps only those; pass
  `visible_only=False` for **every** geometric pass (the radio-operator view), each
  annotated.
- `peak_magnitude` is the brightest magnitude over the visible portion (lower = brighter);
  `None` with no visible portion or no catalogued standard brightness.

`find_passes` streams progress to stderr by default; we pass `progress=False` here to keep
the notebook output tidy.

In [ ]:
passes = pgr.find_passes(
    tle, durham, 2 * 86400, start=start, min_elevation_deg=10, progress=False
)
print(f"{len(passes)} visible passes over 2 days")

## 3. The pass table — `passes_to_dataframe`

`passes_to_dataframe` returns one row per pass with **timezone-aware** datetimes. Pass a
`USTimeZone` member (or any `datetime.tzinfo`) to localize the clock — here US Eastern,
DST-correct for Durham. Pure formatting, no JVM. `duration_s` is `set − rise`; the three
`*_azimuth_deg` columns say where in the sky to look.

In [ ]:
df = pgr.passes_to_dataframe(passes, tz=pgr.USTimeZone.EASTERN)
df

## 4. Export — CSV and a calendar file

Two file exporters, both pure-Python (no JVM):

- `export_passes_csv` writes the same columns as CSV, **UTC only** (the archival
  convention), under a `# key: value` metadata header — reload with
  `pandas.read_csv(path, comment="#")`.
- `export_passes_ics` writes a hand-rolled iCalendar file — one event per pass — that
  imports straight into Google / Apple / Outlook calendars. Timestamps are UTC `Z`; the
  client localizes them. `name=` labels the satellite (a `Pass` carries none).

In [ ]:
import tempfile
from pathlib import Path

out_dir = Path(tempfile.mkdtemp())
pgr.export_passes_csv(passes, out_dir / "iss_passes.csv")
pgr.export_passes_ics(passes, out_dir / "iss_passes.ics", name="ISS")

print("wrote:", ", ".join(sorted(p.name for p in out_dir.iterdir())))
print()
# Show the calendar header + the first event.
print((out_dir / "iss_passes.ics").read_text().split("END:VEVENT")[0] + "END:VEVENT")

## 5. Where to look — `plot_sky_chart`

`plot_sky_chart` draws each pass's arc across the observer's sky (North up, zenith at the
centre, horizon at the rim). A `Pass` stores only scalars, so the arc is **recomputed**
from the TLE, and split into a **sunlit** segment (gold) and an **eclipsed** segment (grey
dashed) — you can see exactly where a pass slips into Earth's shadow. Rise / set carry
local times and azimuths; culmination carries the max elevation and peak magnitude.

This verb is JVM-touching (it re-propagates each arc). The static figure renders inline
with the default backend.

In [ ]:
pgr.plot_sky_chart(tle, durham, passes, tz=pgr.USTimeZone.EASTERN)

## 6. When — `plot_pass_timeline`

The timeline is the "when" view to the sky chart's "where": a wall-clock x-axis, one bar
per pass with height = max elevation, **gold** if the satellite is sunlit at culmination
and **grey-hatched** if it has already slipped into shadow, with the peak magnitude
annotated above. It draws only the stored `Pass` fields — no TLE, no JVM.

In [ ]:
pgr.plot_pass_timeline(passes, tz=pgr.USTimeZone.EASTERN)

---

That's Feature 1.5: **find** visible passes (`find_passes`), **tabulate** them
(`passes_to_dataframe`), **export** to CSV / calendar (`export_passes_csv` /
`export_passes_ics`), and **plot** where and when to look (`plot_sky_chart` /
`plot_pass_timeline`). Set `visible_only=False` for every geometric pass, or
`standard_magnitude=` to override the catalogued brightness. The full contract is in
`docs/features.md` §1.5.